# Conformal Sign-off Evaluation

This notebook demonstrates the split-conformal procedure that wraps the trained policy and produces a calibrated upper bound on residual WNS risk at user-specified confidence $1-\alpha$.

We will:
1. Calibrate a `ConformalSignoff` on synthetic data
2. Inspect the resulting envelope
3. Verify empirical coverage on a held-out set
4. Show what happens under tool-version drift

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve() / 'src'))

import numpy as np
import matplotlib.pyplot as plt
from robin_fpga.conformal import ConformalSignoff

rng = np.random.default_rng(42)

## 1. Synthetic calibration data

Imagine the slack-regression head predicts $\widehat{\mathrm{WNS}}$ with Gaussian noise around the true post-route WNS:

$$\mathrm{WNS}_i = \widehat{\mathrm{WNS}}(z_i) + \varepsilon_i,\quad \varepsilon_i \sim \mathcal{N}(0, 0.15^2)$$

In [ ]:
n_cal = 100
pred_cal = rng.normal(0.3, 0.4, n_cal)
obs_cal  = pred_cal + rng.normal(0, 0.15, n_cal)

signoff = ConformalSignoff(alpha=0.05)
signoff.calibrate(pred_cal, obs_cal)
print(f'Calibration size: {signoff._calibration_size}')
print(f'Conformal quantile q_{{1-alpha}}: {signoff.quantile:.4f} ns')

## 2. Inspect the envelope on a single prediction

The envelope is `[\widehat{WNS}(z) - q, \widehat{WNS}(z) + q]`. The design is accepted iff the lower endpoint is non-negative.

In [ ]:
for wns_pred in [0.50, 0.30, 0.10, -0.05]:
    decision = signoff.evaluate(
        prediction=wns_pred,
        audit_metadata={'tool': 'vivado', 'tool_version': '2024.2', 'seeds': list(range(1, 11))},
    )
    print(f'pred={wns_pred:+.2f}  envelope=[{decision.lower:+.3f}, {decision.upper:+.3f}]  '
          f'accept={decision.accepted}  hash={decision.audit_hash[:12]}')

## 3. Verify coverage on a held-out test set

Under exchangeability we expect empirical coverage ~ $1-\alpha$.

In [ ]:
n_test = 2000
pred_test = rng.normal(0.3, 0.4, n_test)
obs_test  = pred_test + rng.normal(0, 0.15, n_test)

coverage = signoff.empirical_coverage(pred_test, obs_test)
print(f'Target coverage:    {1 - signoff.alpha:.3f}')
print(f'Empirical coverage: {coverage:.3f}')

## 4. Coverage under tool-version drift

We simulate a tool minor-version bump by inflating the test-set noise. Coverage degrades.

In [ ]:
noise_levels = np.linspace(0.10, 0.40, 7)
coverages = []
for sigma in noise_levels:
    p = rng.normal(0.3, 0.4, n_test)
    o = p + rng.normal(0, sigma, n_test)
    coverages.append(signoff.empirical_coverage(p, o))

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(noise_levels, coverages, 'o-', color='#C2410C', label='empirical coverage')
ax.axhline(1 - signoff.alpha, color='gray', ls='--', label='target $1-\\alpha$')
ax.set_xlabel('test-set noise $\\sigma$ (proxy for tool drift)')
ax.set_ylabel('empirical coverage')
ax.set_title('Coverage degrades under distribution shift')
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

## Conclusion

- Under exchangeability, the conformal envelope hits the target coverage within sampling noise.
- Under distribution shift (tool drift, PVT corner extrapolation), coverage degrades — this is the trigger for **recalibration**.
- The audit hash is deterministic; the same `(weights, tool, seeds, corners)` produces the same hash, enabling reproducible sign-off.